In [0]:
%run ../../02_common_utils/operations

In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS staging")
spark.sql("USE SCHEMA staging")

from pyspark.sql.functions import (
    col, trim, lit, when, to_date,
    md5, concat_ws, current_timestamp
)
from pyspark.sql.types import (
    LongType, IntegerType, DecimalType
)
from datetime import datetime

team_name   = "team_lemma"
bronze_db   = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db   = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"

# Default to widget if present, but we will look up the carried_run_id
dbutils.widgets.dropdown("current_batch", "2", ["2", "3"], "Current Batch")
current_batch_val = dbutils.widgets.get("current_batch")
batch_label = f"Batch{current_batch_val}"

# Extract the carried run_id AND batch from the Bronze table
try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {bronze_db}.dailymarket WHERE _batch = '{batch_label}' LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else batch_label
except Exception:
    carried_run_id = "unknown"
    carried_batch = batch_label
print(f"bronze  : {bronze_db}")
print(f"staging : {staging_db}")
print(f"run_id  : {run_id}")

In [0]:
from pyspark.sql.functions import expr

def safe_date(column_name: str, fmt: str = "yyyyMMdd"):
    """
    Null-safe date cast — returns NULL for empty / unparseable strings
    instead of raising CANNOT_PARSE_TIMESTAMP.
    Uses Spark SQL try_to_date() under the hood.
    """
    return expr(f"try_to_date(trim({column_name}), '{fmt}')")


def safe_eff_date():
    return expr("try_to_date(trim(substring(PTS, 1, 8)), 'yyyyMMdd')").alias("EffectiveDate")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_to_staging_market_dailymarket_2_3', f'Starting processing for DailyMarket CDC (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
# dbutils.widgets.dropdown("current_batch", "2", ["2", "3"], "Current Batch")
# current_batch = dbutils.widgets.get("current_batch")
# print(f"Processing DailyMarket CDC for Batch{current_batch}")


In [0]:
batch_label = f"Batch{current_batch}"

incoming = (
    spark.table(f"{bronze_db}.dailymarket")
    .filter(col("_batch") == batch_label)
    .select(
        trim(col("DM_DATE")).alias("DM_DATE"),
        trim(col("DM_S_SYMB")).alias("DM_S_SYMB"),
        trim(col("DM_CLOSE")).alias("DM_CLOSE"),
        trim(col("DM_HIGH")).alias("DM_HIGH"),
        trim(col("DM_LOW")).alias("DM_LOW"),
        trim(col("DM_VOL")).alias("DM_VOL"),
        trim(col("DM_ACTION")).alias("DM_ACTION"),
    )
    .withColumn(
        "row_hash",
        md5(concat_ws("|",
            col("DM_DATE"), col("DM_S_SYMB"),
            col("DM_CLOSE"), col("DM_HIGH"),
            col("DM_LOW"),   col("DM_VOL")
        ))
    )
)

silver_ready = spark.catalog.tableExists(
    f"charles_schwab_retailbrokerage_dev_{team_name}.silver.markethistory"
)
print(f"silver.markethistory exists: {silver_ready}")

if silver_ready:
    silver_mh = (
        spark.table(f"{silver_db}.markethistory")
        .select(
            col("dm_date").alias("s_date"),
            col("dm_s_symb").alias("s_symb"),
            md5(concat_ws("|",
                col("dm_date"),  col("dm_s_symb"),
                col("dm_close"), col("dm_high"),
                col("dm_low"),   col("dm_vol")
            )).alias("silver_hash")
        )
    )
    staged = (
        incoming
        .join(
            silver_mh,
            (incoming["DM_DATE"]   == silver_mh["s_date"]) &
            (incoming["DM_S_SYMB"] == silver_mh["s_symb"]),
            how="left"
        )
        .withColumn("cdc_action",
            when(col("DM_ACTION") == "D",              lit("D"))
           .when(col("silver_hash").isNull(),           lit("N"))
           .when(col("row_hash") != col("silver_hash"), lit("C"))
           .otherwise(                                  lit("X"))
        )
        .drop("s_date", "s_symb", "silver_hash")   
    )
else:
    print(" silver.markethistory not found — all rows marked N")
    staged = incoming.withColumn("cdc_action", lit("N"))

staged_final = (
    staged
    .withColumn("_batch",   lit(batch_label))
    .withColumn("_run_id",  lit(run_id))
    .withColumn("_load_ts", current_timestamp())
)

(staged_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{staging_db}.dailymarket_current"))

print(f"staging.dailymarket_current written for {batch_label}")

In [0]:
result = (
    spark.table(f"{staging_db}.dailymarket_current")
    .groupBy("cdc_action")
    .count()
    .orderBy("cdc_action")
)
display(result)



total = spark.table(f"{staging_db}.dailymarket_current").count()
print(f"\nTotal rows for {batch_label}: {total:,}  (expected 7,360)")
print("Only N and C rows will be MERGEd into silver.markethistory")
print("D rows will be DELETEd from silver.markethistory")
print("X rows will be SKIPped")

In [0]:

source_count = (
    spark.table(f"{bronze_db}.dailymarket")
    .filter(col("_batch") == batch_label)
    .count()
)
target_count = spark.table(f"{staging_db}.dailymarket_current").count()
carried_run_id = str(
    spark.table(f"{staging_db}.dailymarket_current").select("`_run_id`").first()[0]
)

log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id=batch_label,
    domain="MARKET",
    table_name="dailymarket_current",
    source_layer="bronze",
    target_layer="staging",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch=batch_label,
    layer="staging",
    table_name="dailymarket_current",
    operation="OVERWRITE",
    rows_affected=target_count
)

log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_to_staging_market_dailymarket_2_3', 'Successfully completed processing for DailyMarket CDC.')
print(f"source_count : {source_count:,}")   
print(f"target_count : {target_count:,}")   